# Chapter 8: Heteroscedasticity

Topics: heteroscedasticity-robust inference, Breusch-Pagan and White tests, WLS estimation.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)
df = woo.dataWoo('hprice1')

## 8.1 Heteroscedasticity-Robust Inference

In [ ]:
# OLS with and without robust SEs
m_ols    = smf.ols('price ~ lotsize + sqrft + bdrms', data=df).fit()
m_robust = smf.ols('price ~ lotsize + sqrft + bdrms', data=df).fit(cov_type='HC3')

comparison = pd.DataFrame({
    'OLS SE':    m_ols.bse,
    'Robust SE': m_robust.bse,
    'OLS t':     m_ols.tvalues,
    'Robust t':  m_robust.tvalues
})
print("Standard vs Robust Standard Errors:")
print(comparison.round(4))

## 8.2 Heteroscedasticity Tests

In [ ]:
# Breusch-Pagan test
bp_stat, bp_pval, _, _ = het_breuschpagan(m_ols.resid, m_ols.model.exog)
print(f"Breusch-Pagan test: stat={bp_stat:.4f}, p-value={bp_pval:.4f}")
print("→ Reject H0 of homoscedasticity" if bp_pval < 0.05 else "→ Fail to reject H0")

# White test
w_stat, w_pval, _, _ = het_white(m_ols.resid, m_ols.model.exog)
print(f"\nWhite test: stat={w_stat:.4f}, p-value={w_pval:.4f}")

# Residual plot
plt.figure(figsize=(8, 4))
plt.scatter(m_ols.fittedvalues, m_ols.resid, alpha=0.5, color='steelblue')
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Fitted Values'); plt.ylabel('Residuals')
plt.title('Residuals vs Fitted Values')
plt.savefig(f"{output_dir}/fig1_residual_plot.png", dpi=150)
plt.show()

## 8.3 Weighted Least Squares (WLS)

In [ ]:
# Weight by 1/lotsize (proxy for variance)
m_wls = smf.wls('price ~ lotsize + sqrft + bdrms', data=df,
                  weights=1/df['lotsize']).fit()

print("WLS Results:")
print(m_wls.summary())

# Compare OLS vs WLS coefficients
comp = pd.DataFrame({'OLS': m_ols.params, 'WLS': m_wls.params})
print("\nOLS vs WLS Coefficients:")
print(comp.round(4))